# Seção 5 (tipos especializados) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`05_tipos_de_data`](05_tipos_de_data.ipynb) | `PastDatetime` com datetime *naive* compara com a hora **local** — o notebook fixa o fuso para ser reprodutível |
| [`06_tipos_de_rede`](06_tipos_de_rede.ipynb) | `HttpUrl` normaliza a URL (barra final no host) |

## Novas

**1. `HttpUrl` não é `str`.**

In [1]:
from uuid import uuid1, uuid4
from pydantic import BaseModel, HttpUrl, PositiveInt, TypeAdapter, UUID4, ValidationError


class Site(BaseModel):
    url: HttpUrl


site = Site(url="https://example.com")
print(f"type(site.url).__name__: {type(site.url).__name__}")
print(f"site.url == 'https://example.com': {site.url == 'https://example.com'}")
print(f"str(site.url): {str(site.url)}")
try:
    site.url.startswith("https")
except AttributeError as ex:
    print(f"site.url.startswith(...): AttributeError: {ex}")

type(site.url).__name__: HttpUrl
site.url == 'https://example.com': False
str(site.url): https://example.com/
site.url.startswith(...): AttributeError: 'HttpUrl' object has no attribute 'startswith'


Comparar com a string original falha duas vezes: pelo tipo e pela barra final acrescentada. Código que trata a URL como texto
(`startswith`, concatenação, chave de dicionário, coluna do banco) precisa de `str(site.url)`, de preferência num só lugar.

**2. `UUID4` recusa outras versões de UUID.**

In [2]:
for label, value in [("uuid4()", uuid4()), ("uuid1()", uuid1())]:
    try:
        TypeAdapter(UUID4).validate_python(value)
        print(f"{label}: aceito")
    except ValidationError as ex:
        print(f"{label}: {ex.errors()[0]['type']}")

uuid4(): aceito
uuid1(): uuid_version


Ótimo para garantir a origem, ruim para migrações: se outro sistema passar a gerar UUID v7 (ordenável por tempo, cada vez mais
comum em bancos), todo modelo com `UUID4` passa a recusar IDs válidos. Use `UUID4` quando a versão **é** requisito; para "é um
UUID", use `uuid.UUID`.

**3. Tipos "restritos" continuam *lax*: `PositiveInt` aceita `"5"`, `5.0` e `True`.**

In [3]:
for raw in ["5", 5.0, True, 5.5]:
    try:
        print(f"PositiveInt {raw!r}: {TypeAdapter(PositiveInt).validate_python(raw)!r}")
    except ValidationError as ex:
        print(f"PositiveInt {raw!r}: {ex.errors()[0]['type']}")

PositiveInt '5': 5
PositiveInt 5.0: 5
PositiveInt True: 1
PositiveInt 5.5: int_from_float


O tipo restringe o **valor** (positivo), não a **forma** de entrada. Para recusar texto e `bool`, combine com strict:
`Annotated[int, Field(gt=0, strict=True)]`.

## Veja também

- [`pydantic/01_pydantic_basico`](../../01_pydantic_basico.ipynb) (caveats): `EmailStr` normaliza (espaços, domínio em minúsculas)
  e `str` não; `AwareDatetime` para exigir fuso.